# Support Vector Regression (SVR)
Training Linear and RBF SVR models with GridSearch.

In [ ]:
import sys
import os
import pandas as pd
import json

# Add src to path
sys.path.append('../src')

from load_data import load_raw_data
from preprocess import split_data_time_based, preprocess_data
from models import SVRWrapper
from train import train_and_evaluate

In [ ]:
# 1. Load & Preprocess
print("Loading and Preprocessing...")
df = load_raw_data()
train_df, test_df = split_data_time_based(df)
target_col = 'hpi_metro_nsa'

train_df_scaled, test_df_scaled, scaler = preprocess_data(train_df.copy(), test_df.copy(), target_col=target_col)

cols_to_drop = ['year', 'quarter', 'metro_name', 'period_id', target_col]
X_train = train_df_scaled.drop(columns=cols_to_drop, errors='ignore')
y_train = train_df_scaled[target_col]
X_test = test_df_scaled.drop(columns=cols_to_drop, errors='ignore')
y_test = test_df_scaled[target_col]

feature_names = X_train.columns.tolist()

In [ ]:
# 2. Train SVR Linear
print("\n--- SVR Linear ---")
# Simplify grid for speed if needed, but requirements asked for tuning
param_grid_linear = {
    'C': [0.1, 1, 10],
    'epsilon': [0.01, 0.1]
}
svr_linear = SVRWrapper(kernel='linear', param_grid=param_grid_linear)
train_and_evaluate(svr_linear, X_train, y_train, X_test, y_test, "SVR (Linear)")
print(f"Best Params (Linear): {svr_linear.best_params_}")

In [ ]:
# 3. Train SVR RBF
print("\n--- SVR RBF ---")
param_grid_rbf = {
    'C': [1, 10, 100],
    'epsilon': [0.01, 0.1],
    'gamma': ['scale', 0.1]
}
svr_rbf = SVRWrapper(kernel='rbf', param_grid=param_grid_rbf)
train_and_evaluate(svr_rbf, X_train, y_train, X_test, y_test, "SVR (RBF)")
print(f"Best Params (RBF): {svr_rbf.best_params_}")

In [ ]:
# Save best params
params_data = {
    "svr_linear_best_params": svr_linear.best_params_,
    "svr_rbf_best_params": svr_rbf.best_params_
}

os.makedirs('../reports/tables', exist_ok=True)
with open('../reports/tables/svr_params.json', 'w') as f:
    json.dump(params_data, f, indent=4)
print("\nSaved SVR params to reports/tables/svr_params.json")